# Longitudinal Blood Pressure Evolution Figure

This notebook generates a two-panel paper figure:
- Panel A: BP evolution by 1-year mRS category
- Panel B: BP evolution by DCI status

The BP data are filtered to remove measures with concomitant noradrenaline (`noradrenaline_concomitant == 0`).

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from utils.utils import load_encrypted_xlsx, safe_conversion_to_datetime

sns.set_theme(style="whitegrid", context="talk")
pd.options.mode.copy_on_write = True

In [ ]:
# Data paths
registry_path = "/Users/jk1/Library/CloudStorage/OneDrive-UniversitédeGenève/icu_research/dci_sah/data/sos_sah_data/post_hoc_modified_aSAH_DATA_2009_2023_24122023.xlsx"
outcome_data_path = "/Users/jk1/Library/CloudStorage/OneDrive-UniversitédeGenève/icu_research/dci_sah/data/sos_sah_data/follow_up/aSAH_DATA_2009_2024_18122024.xlsx"
bp_path = "/Users/jk1/Library/CloudStorage/OneDrive-UniversitédeGenève/icu_research/dci_sah/data/pdms_data/extracted_data/20240116_SAH_SOS_Blutdruecke.csv"
nor_annotated_bp_path = "/Users/jk1/Library/CloudStorage/OneDrive-UniversitédeGenève/icu_research/dci_sah/data/pdms_data/extracted_data/20240116_SAH_SOS_Blutdruecke_nor_annotated.csv"
registry_pdms_correspondence_path = "/Users/jk1/Library/CloudStorage/OneDrive-UniversitédeGenève/icu_research/dci_sah/data/pdms_data/registry_pdms_correspondence.csv"

# Figure settings
FILTER_NORADRENALINE = True
MAX_DAY = 21
DCI_MAX_DAY = 14
BP_METRIC = "mitteldruck"  # one of: systole, diastole, mitteldruck

required_paths = [
    registry_path,
    outcome_data_path,
    registry_pdms_correspondence_path,
    nor_annotated_bp_path if FILTER_NORADRENALINE else bp_path,
]

missing_paths = [p for p in required_paths if not os.path.exists(p)]
if missing_paths:
    raise FileNotFoundError(
        "Missing required data file(s):\n" + "\n".join(missing_paths)
    )

In [ ]:
# Load data
registry_df = load_encrypted_xlsx(registry_path)
outcome_df = load_encrypted_xlsx(outcome_data_path)
registry_pdms_correspondence_df = pd.read_csv(registry_pdms_correspondence_path)

if FILTER_NORADRENALINE:
    bp_raw_df = pd.read_csv(nor_annotated_bp_path)
    if "noradrenaline_concomitant" not in bp_raw_df.columns:
        raise ValueError("Column 'noradrenaline_concomitant' not found in nor-annotated BP file.")
    bp_df = bp_raw_df.loc[bp_raw_df["noradrenaline_concomitant"] == 0].copy()
else:
    bp_df = pd.read_csv(bp_path, sep=";", decimal=".")

# Basic cleanup
bp_df = bp_df.drop_duplicates(subset=["pNr", "systole", "diastole", "mitteldruck", "timeBd"])
registry_df = registry_df.drop_duplicates().dropna(subset=["SOS-CENTER-YEAR-NO.", "Name", "Date_birth"])

# Harmonize dtypes used for linkage
bp_df = bp_df.merge(registry_pdms_correspondence_df, how="left", on="pNr")
bp_df["Date_birth"] = pd.to_datetime(bp_df["Date_birth"], format="%d.%m.%Y", errors="coerce")
outcome_df["Date_birth"] = pd.to_datetime(outcome_df["Date_birth"], errors="coerce")
registry_df["Date_birth"] = pd.to_datetime(registry_df["Date_birth"], errors="coerce")

# Build mRS at 1y with fallback logic
outcome_df["mRS_FU_1y"] = outcome_df["mRS_FU_1y"].fillna(outcome_df["mRS_2FU_2y"])
outcome_df["mRS_FU_1y"] = outcome_df["mRS_FU_1y"].fillna(outcome_df["mRS_3FU_5y"])
outcome_df["mRS_FU_1y"] = pd.to_numeric(outcome_df["mRS_FU_1y"], errors="coerce")
outcome_df["mRS_discharge"] = pd.to_numeric(outcome_df["mRS_discharge"], errors="coerce")
outcome_df.loc[outcome_df["mRS_discharge"] == 6, "mRS_FU_1y"] = 6

mrs_lookup = outcome_df[["SOS-CENTER-YEAR-NO.", "Name", "Date_birth", "mRS_FU_1y"]].rename(
    columns={"Name": "JoinedName", "mRS_FU_1y": "mrs_1y"}
)
bp_df = bp_df.merge(mrs_lookup, how="left", on=["SOS-CENTER-YEAR-NO.", "JoinedName", "Date_birth"])

bp_df["mrs_1y_02"] = pd.Series(np.where(bp_df["mrs_1y"].between(0, 2, inclusive="both"), "mRS 0-2", "mRS 3-6"), index=bp_df.index)
bp_df.loc[bp_df["mrs_1y"].isna(), "mrs_1y_02"] = np.nan

# Merge DCI information and censor DCI cases after DCI onset
registry_df = registry_df.drop_duplicates(subset=["SOS-CENTER-YEAR-NO.", "Date_birth", "Name"])
registry_df["full_date_dci"] = (
    registry_df["Date_DCI_ischemia_first_image"].astype(str)
    + " "
    + registry_df["Time_DCI_ischemia_first_image"].astype(str)
)
registry_df["full_date_dci"] = registry_df["full_date_dci"].replace("NaT nan", pd.NaT)
registry_df["full_date_dci"] = registry_df["full_date_dci"].apply(safe_conversion_to_datetime)

bp_df = bp_df.merge(
    registry_df[["SOS-CENTER-YEAR-NO.", "Date_birth", "Name", "DCI_ischemia", "full_date_dci"]],
    how="left",
    left_on=["SOS-CENTER-YEAR-NO.", "Date_birth", "JoinedName"],
    right_on=["SOS-CENTER-YEAR-NO.", "Date_birth", "Name"],
)
bp_df["DCI_ischemia"] = pd.to_numeric(bp_df["DCI_ischemia"], errors="coerce")
bp_df["timeBd"] = pd.to_datetime(bp_df["timeBd"], errors="coerce")
bp_df = bp_df.loc[~((bp_df["DCI_ischemia"] == 1) & (bp_df["timeBd"] > bp_df["full_date_dci"]))].copy()

# Relative timeline and daily patient-level medians
first_measure_df = bp_df.groupby("pNr", as_index=False)["timeBd"].min().rename(columns={"timeBd": "first_timeBd"})
bp_df = bp_df.merge(first_measure_df, how="left", on="pNr")
bp_df["relative_time_hours"] = (bp_df["timeBd"] - bp_df["first_timeBd"]).dt.total_seconds() / 3600
bp_df["relative_time_days"] = np.floor(bp_df["relative_time_hours"] / 24)

daily_bp_df = (
    bp_df.groupby(["pNr", "relative_time_days"], as_index=False)
    .agg(
        systole=("systole", "median"),
        diastole=("diastole", "median"),
        mitteldruck=("mitteldruck", "median"),
        mrs_1y_02=("mrs_1y_02", "first"),
        DCI_ischemia=("DCI_ischemia", "first"),
    )
)

daily_bp_df = daily_bp_df.loc[(daily_bp_df["relative_time_days"] >= 0) & (daily_bp_df["relative_time_days"] <= MAX_DAY)].copy()
print(f"Rows in daily summary: {len(daily_bp_df):,}")
print(f"Unique patients: {daily_bp_df['pNr'].nunique():,}")

In [ ]:
def summarize_for_panel(df: pd.DataFrame, group_col: str, metric: str) -> pd.DataFrame:
    out = (
        df.dropna(subset=[group_col, metric])
        .groupby(["relative_time_days", group_col], as_index=False)
        .agg(
            median_bp=(metric, "median"),
            q25=(metric, lambda s: s.quantile(0.25)),
            q75=(metric, lambda s: s.quantile(0.75)),
            n_patients=("pNr", "nunique"),
        )
        .sort_values([group_col, "relative_time_days"])
)
    return out


def plot_panel(ax, summary_df: pd.DataFrame, group_col: str, title: str, palette: dict):
    for group_name, group_df in summary_df.groupby(group_col):
        if pd.isna(group_name):
            continue
        color = palette.get(group_name, None)
        group_df = group_df.sort_values("relative_time_days")
        ax.plot(group_df["relative_time_days"], group_df["median_bp"], label=str(group_name), color=color, linewidth=2.5)
        ax.fill_between(group_df["relative_time_days"], group_df["q25"], group_df["q75"], color=color, alpha=0.2)

    ax.set_title(title, fontsize=14, fontweight="bold")
    ax.set_xlabel("Days since first BP measurement")
    ax.set_ylabel(f"{BP_METRIC} (mmHg)")
    ax.set_xlim(0, MAX_DAY)
    ax.legend(frameon=False, title="Group")


mrs_panel_df = summarize_for_panel(daily_bp_df, "mrs_1y_02", BP_METRIC)

dci_plot_df = daily_bp_df.copy()
dci_plot_df = dci_plot_df.loc[dci_plot_df["relative_time_days"] <= DCI_MAX_DAY].copy()
dci_plot_df["DCI_group"] = dci_plot_df["DCI_ischemia"].map({0: "No DCI", 1: "DCI"})
dci_panel_df = summarize_for_panel(dci_plot_df, "DCI_group", BP_METRIC)

# Version 1: Median trajectory with IQR ribbons (paper default)
fig_v1, axes = plt.subplots(1, 2, figsize=(16, 6), sharey=True)

plot_panel(
    axes[0],
    mrs_panel_df,
    group_col="mrs_1y_02",
    title="Panel A: BP Evolution by mRS",
    palette={"mRS 0-2": "#1b9e77", "mRS 3-6": "#d95f02"},
)

plot_panel(
    axes[1],
    dci_panel_df,
    group_col="DCI_group",
    title="Panel B: BP Evolution by DCI",
    palette={"No DCI": "#1f77b4", "DCI": "#e31a1c"},
)
axes[1].set_xlim(0, DCI_MAX_DAY)

fig_v1.suptitle(
    f"Version 1 - Median BP Trajectory (noradrenaline-filtered, metric={BP_METRIC})",
    fontsize=16,
    y=1.03,
)
fig_v1.tight_layout()
plt.show()

In [ ]:
# Version 4: Distribution-by-window boxplots
mrs_dist_df = daily_bp_df.dropna(subset=["mrs_1y_02", BP_METRIC]).copy()
dci_dist_df = daily_bp_df.loc[daily_bp_df["relative_time_days"] <= DCI_MAX_DAY].copy()
dci_dist_df["DCI_group"] = dci_dist_df["DCI_ischemia"].map({0: "No DCI", 1: "DCI"})
dci_dist_df = dci_dist_df.dropna(subset=["DCI_group", BP_METRIC]).copy()

mrs_dist_df["time_window"] = pd.cut(
    mrs_dist_df["relative_time_days"],
    bins=[-0.1, 3, 7, 14, 21],
    labels=["0-3", "4-7", "8-14", "15-21"],
)
dci_dist_df["time_window"] = pd.cut(
    dci_dist_df["relative_time_days"],
    bins=[-0.1, 3, 7, 14],
    labels=["0-3", "4-7", "8-14"],
)

fig_v4, axes = plt.subplots(1, 2, figsize=(16, 6), sharey=True)

sns.boxplot(
    data=mrs_dist_df,
    x="time_window",
    y=BP_METRIC,
    hue="mrs_1y_02",
    showfliers=False,
    palette={"mRS 0-2": "#1b9e77", "mRS 3-6": "#d95f02"},
    ax=axes[0],
)
axes[0].set_title("Panel A: BP Distribution by mRS and Time Window", fontsize=14, fontweight="bold")
axes[0].set_xlabel("Days since first BP measurement")
axes[0].set_ylabel(f"{BP_METRIC} (mmHg)")
axes[0].legend(title="Group", frameon=False)

sns.boxplot(
    data=dci_dist_df,
    x="time_window",
    y=BP_METRIC,
    hue="DCI_group",
    showfliers=False,
    palette={"No DCI": "#1f77b4", "DCI": "#e31a1c"},
    ax=axes[1],
)
axes[1].set_title("Panel B: BP Distribution by DCI and Time Window", fontsize=14, fontweight="bold")
axes[1].set_xlabel("Days since first BP measurement")
axes[1].set_ylabel(f"{BP_METRIC} (mmHg)")
axes[1].legend(title="Group", frameon=False)

fig_v4.suptitle(f"Version 4 - Time-window Distribution View (metric={BP_METRIC})", fontsize=16, y=1.03)
fig_v4.tight_layout()
plt.show()

In [ ]:
# Version 3: Heatmaps of median BP over time and groups
mrs_heatmap_df = (
    daily_bp_df.dropna(subset=["mrs_1y_02", BP_METRIC])
    .groupby(["relative_time_days", "mrs_1y_02"], as_index=False)[BP_METRIC]
    .median()
)
mrs_heatmap = mrs_heatmap_df.pivot(index="mrs_1y_02", columns="relative_time_days", values=BP_METRIC)

dci_heat_df = daily_bp_df.loc[daily_bp_df["relative_time_days"] <= DCI_MAX_DAY].copy()
dci_heat_df["DCI_group"] = dci_heat_df["DCI_ischemia"].map({0: "No DCI", 1: "DCI"})
dci_heat_df = (
    dci_heat_df.dropna(subset=["DCI_group", BP_METRIC])
    .groupby(["relative_time_days", "DCI_group"], as_index=False)[BP_METRIC]
    .median()
)
dci_heatmap = dci_heat_df.pivot(index="DCI_group", columns="relative_time_days", values=BP_METRIC)

fig_v3, axes = plt.subplots(1, 2, figsize=(16, 5), sharey=False)

sns.heatmap(
    mrs_heatmap,
    cmap="YlGnBu",
    cbar_kws={"label": f"Median {BP_METRIC} (mmHg)"},
    ax=axes[0],
)
axes[0].set_title("Panel A: mRS Heatmap", fontsize=14, fontweight="bold")
axes[0].set_xlabel("Days since first BP measurement")
axes[0].set_ylabel("mRS group")

sns.heatmap(
    dci_heatmap,
    cmap="YlOrRd",
    cbar_kws={"label": f"Median {BP_METRIC} (mmHg)"},
    ax=axes[1],
)
axes[1].set_title("Panel B: DCI Heatmap (Day 0-14)", fontsize=14, fontweight="bold")
axes[1].set_xlabel("Days since first BP measurement")
axes[1].set_ylabel("DCI group")

fig_v3.suptitle(f"Version 3 - Heatmap Representation (metric={BP_METRIC})", fontsize=16, y=1.03)
fig_v3.tight_layout()
plt.show()

In [ ]:
# Version 2: Mean trajectory with 95% CI
mrs_line_df = daily_bp_df.dropna(subset=["mrs_1y_02", BP_METRIC]).copy()
dci_line_df = daily_bp_df.loc[daily_bp_df["relative_time_days"] <= DCI_MAX_DAY].copy()
dci_line_df["DCI_group"] = dci_line_df["DCI_ischemia"].map({0: "No DCI", 1: "DCI"})
dci_line_df = dci_line_df.dropna(subset=["DCI_group", BP_METRIC]).copy()

fig_v2, axes = plt.subplots(1, 2, figsize=(16, 6), sharey=True)

sns.lineplot(
    data=mrs_line_df,
    x="relative_time_days",
    y=BP_METRIC,
    hue="mrs_1y_02",
    estimator="mean",
    errorbar=("ci", 95),
    marker="o",
    linewidth=2,
    palette={"mRS 0-2": "#1b9e77", "mRS 3-6": "#d95f02"},
    ax=axes[0],
)
axes[0].set_title("Panel A: Mean BP by mRS (95% CI)", fontsize=14, fontweight="bold")
axes[0].set_xlabel("Days since first BP measurement")
axes[0].set_ylabel(f"{BP_METRIC} (mmHg)")
axes[0].set_xlim(0, MAX_DAY)
axes[0].legend(title="Group", frameon=False)

sns.lineplot(
    data=dci_line_df,
    x="relative_time_days",
    y=BP_METRIC,
    hue="DCI_group",
    estimator="mean",
    errorbar=("ci", 95),
    marker="o",
    linewidth=2,
    palette={"No DCI": "#1f77b4", "DCI": "#e31a1c"},
    ax=axes[1],
)
axes[1].set_title("Panel B: Mean BP by DCI (95% CI)", fontsize=14, fontweight="bold")
axes[1].set_xlabel("Days since first BP measurement")
axes[1].set_ylabel(f"{BP_METRIC} (mmHg)")
axes[1].set_xlim(0, DCI_MAX_DAY)
axes[1].legend(title="Group", frameon=False)

fig_v2.suptitle(f"Version 2 - Mean BP with 95% CI (metric={BP_METRIC})", fontsize=16, y=1.03)
fig_v2.tight_layout()
plt.show()

In [ ]:
# Version 5: Publication-ready legacy style (day-wise boxplots by metric, styled like V6)
metrics_to_plot = [
    ("systole", "Systolic BP (mmHg)"),
    ("diastole", "Diastolic BP (mmHg)"),
    ("mitteldruck", "MAP (mmHg)"),
]
BOX_ALPHA_V5 = 0.55

mrs_old_df = daily_bp_df.dropna(subset=["mrs_1y_02"]).copy()
dci_old_df = daily_bp_df.loc[daily_bp_df["relative_time_days"] <= DCI_MAX_DAY].copy()
dci_old_df["DCI_group"] = dci_old_df["DCI_ischemia"].map({0: "No DCI", 1: "DCI"})
dci_old_df = dci_old_df.dropna(subset=["DCI_group"]).copy()

mrs_palette_v5 = {"mRS 0-2": "#FF2F92", "mRS 3-6": "#8B6BB8"}
dci_palette_v5 = {"No DCI": "#E7B8A8", "DCI": "#59A8A8"}

fig_v5, axes = plt.subplots(3, 2, figsize=(18, 12), sharex=False)

for row_idx, (metric, y_label) in enumerate(metrics_to_plot):
    ax_left = axes[row_idx, 0]
    ax_right = axes[row_idx, 1]

    sns.boxplot(
        data=mrs_old_df,
        x="relative_time_days",
        y=metric,
        hue="mrs_1y_02",
        palette=mrs_palette_v5,
        showfliers=False,
        linewidth=0.8,
        width=0.75,
        ax=ax_left,
    )

    sns.boxplot(
        data=dci_old_df,
        x="relative_time_days",
        y=metric,
        hue="DCI_group",
        palette=dci_palette_v5,
        showfliers=False,
        linewidth=0.8,
        width=0.75,
        ax=ax_right,
    )

    ax_left.set_xlim(-0.5, MAX_DAY + 0.5)
    ax_right.set_xlim(-0.5, DCI_MAX_DAY + 0.5)
    ax_left.set_ylim(bottom=0)
    ax_right.set_ylim(bottom=0)

    ax_left.set_ylabel(y_label)
    ax_right.set_ylabel("")
    ax_left.set_xlabel("Time from admission (days)" if row_idx == 2 else "")
    ax_right.set_xlabel("Time from admission (days)" if row_idx == 2 else "")

    if row_idx == 0:
        ax_left.legend(frameon=False, loc="lower right", title=None)
        ax_right.legend(frameon=False, loc="lower right", title=None)
    else:
        if ax_left.get_legend() is not None:
            ax_left.get_legend().remove()
        if ax_right.get_legend() is not None:
            ax_right.get_legend().remove()

# Apply alpha and style cleanup across all panels
for r in range(3):
    for c in range(2):
        ax = axes[r, c]
        for patch in ax.artists:
            patch.set_alpha(BOX_ALPHA_V5)
        for patch in ax.patches:
            patch.set_alpha(BOX_ALPHA_V5)
        ax.grid(True, axis="both", alpha=0.3, linewidth=0.8)
        ax.spines["top"].set_visible(False)
        ax.spines["right"].set_visible(False)
        ax.tick_params(axis="x", labelrotation=45)
        xticks = ax.get_xticks()
        ax.set_xticks(xticks)
        ax.set_xticklabels([str(int(t)) if float(t).is_integer() else f"{t:g}" for t in xticks])

fig_v5.tight_layout()
plt.show()

In [ ]:
# Version 6: Publication-ready daily boxplots for median BP only (day 0-14 for both panels)
MRS_MAX_DAY_V6 = 14
day_order_v6 = list(range(0, MRS_MAX_DAY_V6 + 1))
BOX_ALPHA_V6 = 0.55

mrs_v6_df = daily_bp_df.loc[daily_bp_df["relative_time_days"] <= MRS_MAX_DAY_V6].copy()
mrs_v6_df = mrs_v6_df.dropna(subset=["mrs_1y_02", "mitteldruck"]).copy()
mrs_v6_df["relative_time_days"] = mrs_v6_df["relative_time_days"].astype(int)

dci_v6_df = daily_bp_df.loc[daily_bp_df["relative_time_days"] <= DCI_MAX_DAY].copy()
dci_v6_df["DCI_group"] = dci_v6_df["DCI_ischemia"].map({0: "No DCI", 1: "DCI"})
dci_v6_df = dci_v6_df.dropna(subset=["DCI_group", "mitteldruck"]).copy()
dci_v6_df["relative_time_days"] = dci_v6_df["relative_time_days"].astype(int)

# Palette matched to provided examples (ROC-style for mRS, boxplot-style for DCI)
mrs_palette_v6 = {"mRS 0-2": "#FF2F92", "mRS 3-6": "#8B6BB8"}
dci_palette_v6 = {"No DCI": "#E7B8A8", "DCI": "#59A8A8"}

fig_v6, axes = plt.subplots(1, 2, figsize=(16, 6), sharey=True)

sns.boxplot(
    data=mrs_v6_df,
    x="relative_time_days",
    y="mitteldruck",
    hue="mrs_1y_02",
    order=day_order_v6,
    palette=mrs_palette_v6,
    showfliers=False,
    linewidth=0.8,
    width=0.75,
    ax=axes[0],
)
axes[0].set_xlabel("Time from admission (days)")
axes[0].set_ylabel("MAP (mmHg)")
axes[0].legend(frameon=False, loc="lower right", title=None)

sns.boxplot(
    data=dci_v6_df,
    x="relative_time_days",
    y="mitteldruck",
    hue="DCI_group",
    order=day_order_v6,
    palette=dci_palette_v6,
    showfliers=False,
    linewidth=0.8,
    width=0.75,
    ax=axes[1],
)
axes[1].set_xlabel("Time from admission (days)")
axes[1].set_ylabel("MAP (mmHg)")
axes[1].legend(frameon=False, loc="lower right", title=None)

for ax in axes:
    for patch in ax.artists:
        patch.set_alpha(BOX_ALPHA_V6)
    for patch in ax.patches:
        patch.set_alpha(BOX_ALPHA_V6)
    ax.set_xticks(range(0, 15, 2))
    ax.set_xlim(-0.5, 14.5)
    ax.set_ylim(bottom=0)
    ax.grid(True, axis="both", alpha=0.3, linewidth=0.8)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    xticks = ax.get_xticks()
    ax.set_xticks(xticks)
    ax.set_xticklabels([str(int(t)) if float(t).is_integer() else f"{t:g}" for t in xticks])

fig_v6.tight_layout()
plt.show()

In [ ]:
# Optional: save all figure versions for manuscript use
output_dir = "/Users/jk1/temp/bp_dci/bp_trajectories"
os.makedirs(output_dir, exist_ok=True)

save_targets = [
    ("figure_v1_median_iqr_nor_filtered.png", "fig_v1"),
    ("figure_v2_mean_ci_nor_filtered.png", "fig_v2"),
    ("figure_v3_heatmap_nor_filtered.png", "fig_v3"),
    ("figure_v4_timewindow_boxplot_nor_filtered.png", "fig_v4"),
    ("figure_v5_legacy_boxplot_refined_nor_filtered.png", "fig_v5"),
    ("figure_v6_median_bp_daily_boxplot_day14_nor_filtered.png", "fig_v6"),
]

for fname, fig_name in save_targets:
    if fig_name in globals():
        fig_obj = globals()[fig_name]
        fig_obj.savefig(os.path.join(output_dir, fname), dpi=300, bbox_inches="tight")
        print(f"Saved: {os.path.join(output_dir, fname)}")
    else:
        print(f"Skipped {fname}: {fig_name} not found in current kernel state")